## Silver Layer to Daily Stats Table

In [1]:
import polars as pl
from polars import col

import duckdb

### Ingest from Silver

In [2]:
con = duckdb.connect("../burger.db")

df_silver = con.execute("SELECT * FROM silver").pl()

### Transformations

In [3]:
df_gold = df_silver.group_by('loaded_at').agg(
    col('total_cost').mean().round(2).alias('average_total_cost'),
    col('total_cost').mean().round(2).alias('median_total_cost'),
    col('total_cost').min().alias('lowest_total_cost'),
    col('item_id').count().alias('sample_size')
).sort('loaded_at')

display(df_gold)

loaded_at,average_total_cost,median_total_cost,lowest_total_cost,sample_size
date,f64,f64,f64,u32
2026-07-18,106.54,106.54,103.98,46
2026-07-20,112.65,112.65,95.98,147


### Load to gold_summary

In [4]:
con.execute("CREATE OR REPLACE TABLE gold_summary AS SELECT * FROM df_gold")

con.close()